# Laboratorio 8 - Evidencia Fase 1 y Ejercicio 3

Este notebook evidencia la descarga inicial de datos 2026 y la exploracion directa de archivos Parquet con DuckDB.

## Objetivos

- Verificar que los archivos Parquet descargados existen localmente.
- Consultar los archivos directamente con DuckDB, sin importarlos a tablas.
- Obtener conteos, esquema, muestra de registros y problemas iniciales de calidad.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

con = duckdb.connect()
raw_path = Path('/workspace/data/raw')
files = sorted(raw_path.glob('**/*.parquet'))
print(f'Archivos Parquet encontrados: {len(files)}')
for file in files:
    print(file.relative_to('/workspace'))

Archivos Parquet encontrados: 16
data/raw/green/2026/green_tripdata_2026-01.parquet
data/raw/green/2026/green_tripdata_2026-02.parquet
data/raw/green/2026/green_tripdata_2026-03.parquet
data/raw/green/2026/green_tripdata_2026-04.parquet
data/raw/green/2026/green_tripdata_2026-05.parquet
data/raw/green/2026/green_tripdata_2026-06.parquet
data/raw/green/2026/green_tripdata_2026-07.parquet
data/raw/green/2026/green_tripdata_2026-08.parquet
data/raw/yellow/2026/yellow_tripdata_2026-01.parquet
data/raw/yellow/2026/yellow_tripdata_2026-02.parquet
data/raw/yellow/2026/yellow_tripdata_2026-03.parquet
data/raw/yellow/2026/yellow_tripdata_2026-04.parquet
data/raw/yellow/2026/yellow_tripdata_2026-05.parquet
data/raw/yellow/2026/yellow_tripdata_2026-06.parquet
data/raw/yellow/2026/yellow_tripdata_2026-07.parquet
data/raw/yellow/2026/yellow_tripdata_2026-08.parquet


## Conteo de archivos por tipo de taxi

In [2]:
con.execute("""
SELECT taxi_type, COUNT(DISTINCT filename) AS files
FROM (
    SELECT 'yellow' AS taxi_type, filename
    FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet', filename = true)
    UNION ALL
    SELECT 'green' AS taxi_type, filename
    FROM read_parquet('/workspace/data/raw/green/2026/*.parquet', filename = true)
)
GROUP BY taxi_type
ORDER BY taxi_type
""").df()

,taxi_type,files
0,green,8
1,yellow,8


## Conteo de registros

In [3]:
con.execute("""
SELECT 'yellow' AS taxi_type, COUNT(*) AS rows
FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet')
UNION ALL
SELECT 'green' AS taxi_type, COUNT(*) AS rows
FROM read_parquet('/workspace/data/raw/green/2026/*.parquet')
ORDER BY taxi_type
""").df()

,taxi_type,rows
0,green,337114
1,yellow,29703355


## Esquemas de los archivos

In [4]:
yellow_schema = con.execute("DESCRIBE SELECT * FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet')").df()
green_schema = con.execute("DESCRIBE SELECT * FROM read_parquet('/workspace/data/raw/green/2026/*.parquet')").df()
display(yellow_schema)
display(green_schema)

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


## Muestra de registros

In [5]:
sample = con.execute("""
SELECT * FROM (
    SELECT 'yellow' AS taxi_type, VendorID, tpep_pickup_datetime AS pickup_datetime,
           tpep_dropoff_datetime AS dropoff_datetime, passenger_count,
           trip_distance, total_amount, payment_type
    FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet')
    LIMIT 5
)
UNION ALL
SELECT * FROM (
    SELECT 'green' AS taxi_type, VendorID, lpep_pickup_datetime AS pickup_datetime,
           lpep_dropoff_datetime AS dropoff_datetime, passenger_count,
           trip_distance, total_amount, payment_type
    FROM read_parquet('/workspace/data/raw/green/2026/*.parquet')
    LIMIT 5
)
""").df()
sample

,taxi_type,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,total_amount,payment_type
0,yellow,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,15.86,1
1,yellow,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,13.65,2
2,yellow,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,18.95,1
3,yellow,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,55.56,1
4,yellow,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,23.10,1
5,green,1,2026-01-01 00:27:58,2026-01-01 00:55:16,2,6.20,45.20,1
6,green,2,2026-01-01 00:44:33,2026-01-01 01:32:56,5,5.36,61.20,1
7,green,1,2026-01-01 00:23:45,2026-01-01 00:45:03,4,10.60,46.00,1
8,green,1,2026-01-01 00:44:33,2026-01-01 01:00:45,1,4.20,25.05,2
9,green,2,2026-01-01 00:46:04,2026-01-01 01:04:40,1,2.76,21.60,2


## Revision inicial de calidad

In [6]:
quality = con.execute("""
SELECT 'yellow' AS taxi_type,
       COUNT(*) AS rows,
       SUM(passenger_count IS NULL) AS null_passenger_count,
       SUM(passenger_count <= 0) AS non_positive_passengers,
       SUM(trip_distance < 0) AS negative_distance,
       SUM(trip_distance = 0) AS zero_distance,
       SUM(total_amount < 0) AS negative_total,
       SUM(tpep_dropoff_datetime < tpep_pickup_datetime) AS dropoff_before_pickup
FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet')
UNION ALL
SELECT 'green' AS taxi_type,
       COUNT(*) AS rows,
       SUM(passenger_count IS NULL) AS null_passenger_count,
       SUM(passenger_count <= 0) AS non_positive_passengers,
       SUM(trip_distance < 0) AS negative_distance,
       SUM(trip_distance = 0) AS zero_distance,
       SUM(total_amount < 0) AS negative_total,
       SUM(lpep_dropoff_datetime < lpep_pickup_datetime) AS dropoff_before_pickup
FROM read_parquet('/workspace/data/raw/green/2026/*.parquet')
ORDER BY taxi_type
""").df()
quality

,taxi_type,rows,null_passenger_count,non_positive_passengers,negative_distance,zero_distance,negative_total,dropoff_before_pickup
0,green,337114,48775.0,4527.0,0.0,12212.0,1023.0,5.0
1,yellow,29703355,7716688.0,91359.0,0.0,952231.0,161835.0,10.0


## Interpretacion inicial

- El conjunto local contiene archivos publicados de enero a agosto de 2026 para taxis amarillos y verdes.
- Los taxis amarillos tienen un volumen de registros mucho mayor que los taxis verdes.
- Se observan problemas iniciales de calidad: pasajeros nulos o no positivos, viajes con distancia cero, montos negativos y algunos registros con hora de llegada anterior a la salida.
- Para combinar taxis amarillos y verdes se deben normalizar nombres de columnas de fecha, porque usan prefijos diferentes (`tpep` y `lpep`).